# STT – Version 4 : semi temps réel robuste + arrêt vocal

Cette version améliore la V2/V3 avec les objectifs suivants :

- rendre le comportement **plus robuste** au bruit et aux silences,
- utiliser une **mesure de bruit de fond** pour adapter le seuil de silence,
- normaliser le signal audio avant transcription,
- supporter plusieurs **mots-clés d'arrêt vocal** (ex : "stop", "arrête", "fin"),
- conserver un **historique du texte reconnu** au fil des segments.

Cette V4 est pensée comme une version plus réaliste, prête à être reliée
au pipeline « Texte → Signes ».


In [1]:
import os
import shutil
import numpy as np
import sounddevice as sd
import whisper

# (Optionnel) S'assurer que ffmpeg est dans le PATH si besoin
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"
print("ffmpeg vu par Python ? ->", shutil.which("ffmpeg"))

# Paramètres audio
FS = 16000            # fréquence d'échantillonnage
SEGMENT_DURATION = 2  # durée d'un segment (s)
N_SAMPLES = FS * SEGMENT_DURATION

# Mots-clés d'arrêt vocal (en minuscules)
STOP_WORDS = ["stop", "arrête", "arrete", "fin"]


ffmpeg vu par Python ? -> C:\ffmpeg\ffmpeg-8.0-essentials_build\bin\ffmpeg.EXE


In [2]:
model = whisper.load_model("small")
print("Modèle Whisper chargé (V4).")


Modèle Whisper chargé (V4).


In [3]:
def compute_energy(signal: np.ndarray) -> float:
    """
    Calcule l'énergie moyenne absolue d'un signal mono.
    """
    return float(np.mean(np.abs(signal)))


def normalize_audio(signal: np.ndarray) -> np.ndarray:
    """
    Normalise le signal dans [-1, 1] en évitant les divisions inutiles.
    """
    max_val = np.max(np.abs(signal)) if signal.size > 0 else 0.0
    if max_val < 1e-6:
        return signal
    return signal / max_val


def is_silence(signal: np.ndarray, base_energy: float, factor: float = 3.0, min_threshold: float = 0.005) -> bool:
    """
    Détermine si un segment est silencieux en comparant son énergie
    à une énergie de base (bruit de fond) multipliée par un facteur.

    - base_energy : énergie du bruit de fond mesurée au début
    - factor : facteur multiplicatif de sensibilité (plus grand => plus strict)
    - min_threshold : seuil minimum absolu pour éviter trop de sensibilité
    """
    energy = compute_energy(signal)
    dynamic_threshold = max(base_energy * factor, min_threshold)
    print(f"Énergie segment = {energy:.5f} | Seuil dynamique = {dynamic_threshold:.5f}")
    return energy < dynamic_threshold


In [4]:
def capture_and_transcribe_loop_v4(language: str = "fr") -> None:
    """
    Version 4 du STT semi temps réel :
    - estimation du bruit de fond au démarrage,
    - seuil de silence dynamique,
    - normalisation des segments,
    - arrêt vocal avec plusieurs mots-clés,
    - accumulation du texte reconnu.
    """
    print("=== V4 : STT semi temps réel robuste ===")
    print("Parle par petites phrases (2–3 s).")
    print(f"Arrêt vocal possible avec : {', '.join(STOP_WORDS)}")
    print("Interruption manuelle possible avec Ctrl+C.\n")

    # 1) Mesure du bruit de fond
    input("Reste silencieux et appuie sur Entrée pour mesurer le bruit de fond...")
    print("Mesure du bruit de fond (1 seconde de silence)...")

    silence_audio = sd.rec(int(FS * 1), samplerate=FS, channels=1, dtype="float32")
    sd.wait()
    silence_mono = silence_audio[:, 0]
    base_energy = compute_energy(silence_mono)
    print(f"Énergie de bruit de fond mesurée : {base_energy:.5f}\n")

    segment_idx = 0
    full_transcript = []  # Historique des segments reconnus

    try:
        while True:
            print(f"\n[Segment {segment_idx}] Enregistrement {SEGMENT_DURATION} s…")
            audio = sd.rec(int(N_SAMPLES), samplerate=FS, channels=1, dtype="float32")
            sd.wait()

            audio_mono = audio[:, 0]

            # Normalisation du segment
            audio_norm = normalize_audio(audio_mono)

            # Vérification du silence avec seuil dynamique
            if is_silence(audio_norm, base_energy):
                print("[Segment ignoré : silence ou bruit de fond]")
                segment_idx += 1
                continue

            print("[Transcription en cours…]")
            result = model.transcribe(audio_norm, language=language, fp16=False)
            text = (result.get("text") or "").strip()

            if text:
                print(f"[Segment {segment_idx}] Texte reconnu : {text}")
                full_transcript.append(text)

                # Arrêt vocal : si un des mots-clés est dans le texte
                lower_text = text.lower()
                if any(stop_word in lower_text for stop_word in STOP_WORDS):
                    print("Mot-clé d'arrêt détecté dans la transcription.")
                    break

                # 👉 Ici, dans le futur, tu pourras envoyer 'text' au module texte → signes
                # send_to_sign_module(text)

            else:
                print(f"[Segment {segment_idx}] Aucun texte exploitable reconnu.")

            # Affichage de la transcription cumulée
            if full_transcript:
                print("\n[Transcription cumulée jusqu'ici] :")
                print(" ".join(full_transcript))

            segment_idx += 1

    except KeyboardInterrupt:
        print("\nInterruption manuelle (Ctrl+C).")

    print("\n=== Fin de la V4 : système arrêté proprement. ===")
    if full_transcript:
        print("Transcription finale cumulée :")
        print(" ".join(full_transcript))


In [5]:
capture_and_transcribe_loop_v4(language="fr")


=== V4 : STT semi temps réel robuste ===
Parle par petites phrases (2–3 s).
Arrêt vocal possible avec : stop, arrête, arrete, fin
Interruption manuelle possible avec Ctrl+C.

Mesure du bruit de fond (1 seconde de silence)...
Énergie de bruit de fond mesurée : 0.00375


[Segment 0] Enregistrement 2 s…
Énergie segment = 0.10360 | Seuil dynamique = 0.01124
[Transcription en cours…]
[Segment 0] Texte reconnu : ...

[Transcription cumulée jusqu'ici] :
...

[Segment 1] Enregistrement 2 s…
Énergie segment = 0.08743 | Seuil dynamique = 0.01124
[Transcription en cours…]
[Segment 1] Texte reconnu : ...

[Transcription cumulée jusqu'ici] :
... ...

[Segment 2] Enregistrement 2 s…
Énergie segment = 0.06332 | Seuil dynamique = 0.01124
[Transcription en cours…]
[Segment 2] Texte reconnu : ...

[Transcription cumulée jusqu'ici] :
... ... ...

[Segment 3] Enregistrement 2 s…
Énergie segment = 0.06845 | Seuil dynamique = 0.01124
[Transcription en cours…]
[Segment 3] Texte reconnu : Arment QUI

[Transcr

## Différences entre la V3 et la V4

Par rapport à la version V3, cette V4 apporte plusieurs améliorations :

1. **Mesure du bruit de fond**
   - Au démarrage, on enregistre 1 seconde de silence pour estimer l'énergie du bruit de fond.
   - Le seuil de silence devient **dynamique** : il dépend de ce bruit de fond.

2. **Normalisation du signal**
   - Chaque segment audio est normalisé, ce qui rend la détection de silence plus stable.
   - Le modèle Whisper reçoit des amplitudes plus homogènes.

3. **Détection de silence améliorée**
   - La fonction `is_silence` compare l'énergie du segment à un seuil dynamique
     basé sur le bruit de fond *et* un seuil minimal.

4. **Arrêt vocal plus robuste**
   - Plusieurs mots-clés possibles : "stop", "arrête", "arrete", "fin".
   - Le texte est converti en minuscules avant comparaison.

5. **Transcription cumulée**
   - La V4 garde un historique des segments reconnus (`full_transcript`),
   - et affiche à chaque itération la transcription cumulée.

Cette V4 est donc plus réaliste et plus robuste, tout en restant exécutable
sur CPU dans un environnement étudiant.
